# Text Prediction with AutoGluon

`MultiModalPredictor` handles text classification by fine-tuning a pre-trained transformer (e.g. DistilBERT) on your labelled data. You get state-of-the-art NLP with a familiar sklearn-style API.

This notebook covers:
1. Data format: text in a DataFrame column
2. Training a text classifier
3. Making predictions and checking probabilities
4. Choosing a different backbone model
5. Practical gotchas

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. Load Data

AutoGluon detects text columns automatically by inspecting cell values: variable-length strings are treated as text. Short strings (e.g. `'cat'`, `'dog'`) are treated as categorical. The column name does not matter.

In [ ]:
from autogluon.tabular import TabularDataset

# Stanford Sentiment Treebank (SST-2) — binary sentiment classification
train_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/SST/train.csv')
test_data  = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/SST/dev.csv')

label = 'label'
print(f'Train: {train_data.shape}, Test: {test_data.shape}')
print('\nLabel distribution:\n', train_data[label].value_counts())
train_data.head()

In [ ]:
# Inspect a few examples
for _, row in train_data.head(3).iterrows():
    sentiment = 'POSITIVE' if row[label] == 1 else 'NEGATIVE'
    print(f'[{sentiment}] {row["sentence"][:80]}...')
    print()

## 2. Train the Classifier

AutoGluon automatically:
- Detects the `'sentence'` column as text
- Downloads a pre-trained HuggingFace checkpoint
- Fine-tunes it with early stopping

> **GPU is strongly recommended.** On CPU, transformer fine-tuning can take 10× longer.

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    label=label,
    problem_type='binary',   # 'binary', 'multiclass', 'regression'
    path='./ag_text_model',
)

predictor.fit(
    train_data=train_data,
    time_limit=120,          # 2 minutes — enough for a quick fine-tune
)

## 3. Predict and Evaluate

In [ ]:
y_pred = predictor.predict(test_data)
print('Predicted labels (first 5):', y_pred[:5].tolist())

y_prob = predictor.predict_proba(test_data)
print('\nProbability output (first 3 rows):')
print(y_prob.head(3))

In [ ]:
metrics = predictor.evaluate(test_data)
print('Test metrics:', metrics)

## 4. Choosing a Different Backbone

The default backbone is `google/electra-base-discriminator`. You can swap in any HuggingFace text model. Smaller models train faster; larger models are more accurate.

Common choices:
- `'distilbert-base-uncased'` — fast, lightweight
- `'bert-base-uncased'` — classic baseline
- `'roberta-base'` — generally stronger than BERT
- `'google/electra-small-discriminator'` — tiny but surprisingly good

In [ ]:
# Example: switch to a lightweight DistilBERT
fast_predictor = MultiModalPredictor(
    label=label,
    problem_type='binary',
    path='./ag_text_distilbert',
)
fast_predictor.fit(
    train_data=train_data,
    time_limit=120,
    hyperparameters={
        'model.hf_text.checkpoint_name': 'distilbert-base-uncased',
    },
)
print('DistilBERT metrics:', fast_predictor.evaluate(test_data))

## 5. Predicting on Raw Strings

You can also call `.predict()` directly on a DataFrame you build yourself — useful for quick ad-hoc predictions.

In [ ]:
import pandas as pd

my_sentences = pd.DataFrame({
    'sentence': [
        'This movie was absolutely fantastic!',
        'Terrible, boring, and a waste of time.',
        'It was okay, nothing special.',
    ]
})

preds = predictor.predict(my_sentences)
probs = predictor.predict_proba(my_sentences)

for sentence, pred, (_, prob_row) in zip(my_sentences['sentence'], preds, probs.iterrows()):
    print(f'  Text: "{sentence[:50]}"')
    print(f'  Pred: {pred} | Confidence: {prob_row.max():.2%}\n')

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Very long texts get truncated** | BERT-family models cap at 512 tokens (~380 words); text beyond that is silently cut | Summarise long documents beforehand, or use a long-context model like `longformer-base-4096` |
| **Short strings treated as categorical** | Columns with values like `'A'`, `'B'`, `'C'` are not treated as text | Use `column_types={'col': 'text'}` to force text treatment |
| **GPU not detected** | Training falls back to CPU and becomes very slow | Check `torch.cuda.is_available()` and install the GPU-enabled PyTorch wheel |
| **HuggingFace model cache** | Large model weights (500 MB+) are downloaded on first use | Pre-download in your setup script; cache lives in `~/.cache/huggingface` |
| **Class labels are stored as-is** | If labels are `0`/`1` integers, `predict_proba` columns are `0` and `1` (integers), not strings | Access as `y_prob[1]` not `y_prob['1']` |
| **Multiclass with many classes** | Training is slower; small classes (< ~5 examples) may never be predicted | Merge rare classes or use `min_class_count` filtering |
| **Non-UTF-8 text** | Encoding errors on load | Read CSVs with `encoding='utf-8'` or `encoding='latin-1'` as appropriate |